In [56]:
import math
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

In [57]:
class Value:
    # Value类，打算实现图的自动建立，标签和数据的运算的运算方法，还有梯度的反向传播的自动计算，这些目前都已实现
    #运算方法如下，还有一些未实现的
    #已实现的有： 加法（包括交换律可交换，并且可处理浮点类型的类 ），乘法（和加法一样，可处理浮点类型），tanh函数。
    #想要实现的还有：除法包括反向除法（✅），幂运算（✅），减法，和指数函数比如e的x次方（✅），log函数，三角函数（这个可以用指数函数做到）
    
    #初始化的一些参数，data是数据，label是标签，_children是子节点，_op是运算符，grad是梯度，_backward是反向传播的函数，注意初始化的时候设置的是一个空函数
    def __init__(self,data,label='',_children=(),_op=''):
        self.data=data
        self._prev=set(_children)
        self.label=label
        self.grad=0.0
        self._op=_op
        self._backward=lambda:None
    #打印函数，规定print(value)时的输出格式
    def __repr__(self):
        return f"Value(data={self.data})"
    
    #加法函数，规定了正向的value对象加value对象的加法运算，同时定义了反向传播的梯度计算
    def __add__(self,other):
        other=other if isinstance(other, Value) else Value(other)
        out=Value(self.data+other.data,_children=(self,other),_op='+')
        def _backward():
            self.grad+=out.grad*1.0
            other.grad+=out.grad*1.0
        out._backward=_backward
        return out
    #反向加法，当value对象在加法运算中处于右边时，调用该函数
    def __radd__(self,other):
            return self+other

    #减法
    def __sub__(self,other):
        other=other if isinstance(other, Value) else Value(other)
        out=Value(self.data-other.data,_children=(self,other),_op='-')
        def _backward():
            self.grad+=out.grad*1.0
            other.grad+=out.grad*-1.0
        out._backward=_backward
        return out
    #反向减法
    def __rsub__(self,other):
        return other+self*-1
    #乘法函数，实现乘法及其相应的反向传播
    def __mul__(self,other):
        other=other if isinstance(other, Value) else Value(other)
        out=Value(self.data*other.data,_children=(self,other),_op='*')
        def _backward():
            self.grad+=out.grad*other.data
            other.grad+=out.grad*self.data
        out._backward=_backward
        return out
    #反向乘法
    def __rmul__(self,other):
        return self*other

    #除法,目前完成状态✅，仔细想了想除法是乘以倒数，所以可以当成乘法，但是倒数涉及到x的幂运算，所以要先实现幂运算
    def __truediv__(self,other):
        other=other if isinstance(other, Value) else Value(other)
        out=Value(self.data*(other.data**-1),_children=(self,other),_op='/')
        def _backward():
            self.grad+=out.grad*(other.data**-1)
            other.grad+=out.grad*(-self.data*(other.data**-2))
        out._backward=_backward
        return out
    #反向除法，这里可以写简便一点，直接套用乘法和幂运算的组合
    def __rtruediv__(self,other):
        return other*self**-1
        
    #接下来开始实现指数函数，e的x次方，定义了正向计算和反向传播
    def exp(self):
        x=self.data
        out=Value(math.exp(x),_children=(self,),_op='exp')
        def _backward():
            self.grad+=out.grad*math.exp(x)
        out._backward=_backward
        return out
    #成功实现

    #幂运算
    def __pow__(self,other):
        assert isinstance(other,(int,float)), "only supporting int/float powers for now"
        out=Value(self.data**other,_children=(self,),_op=f'**{other}')
        def _backward():
            self.grad+=out.grad*other*(self.data**(other-1))
        out._backward=_backward
        return out

    #双曲tanh函数，定义了tanh函数的正向计算和反向传播
    def tanh(self):
        x=self.data
        t=(math.exp(2*x)-1)/(math.exp(2*x)+1)
        out=Value(t,_children=(self,),_op='tanh')
        def _backward():
            self.grad+=out.grad*(1-t**2)
        out._backward=_backward
        return out

    #反向传播函数，包括了图的DFS算法，拓扑排序，这样就可以保证传播的时候梯度顺序不搞错
    def backward(self):
        topo=[]
        visited=set()
        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build_topo(child)
                topo.append(v)
        build_topo(self)
        self.grad=1.0
        for node in reversed(topo):
            node._backward()


In [58]:
from graphviz import Digraph

def trace(root):
    # builds a set of all nodes and edges in a graph
    nodes, edges = set(), set()
    def build(v):
        if v not in nodes:
            nodes.add(v)
            for child in v._prev:
                edges.add((child, v))
                build(child)
    build(root)
    return nodes, edges

def draw_dot(root):
    dot = Digraph(format='svg', graph_attr={'rankdir': 'LR'})  # LR = left to right

    nodes, edges = trace(root)
    for n in nodes:
        uid = str(id(n))
        # for any value in the graph, create a rectangular ('record') node for it
        dot.node(name=uid, label="{ %s | data %.4f | grad %.4f }" % (n.label, n.data, n.grad), shape='record')
        if n._op:
            # if this value is a result of some operation, create an op node for it
            dot.node(name=uid + n._op, label=n._op)
            # and connect this node to it
            dot.edge(uid + n._op, uid)

    for n1, n2 in edges:
        # connect n1 to the op node of n2
        dot.edge(str(id(n1)), str(id(n2)) + n2._op)

    return dot


In [59]:
def checking():
    #测试函数，测试加法，乘法，tanh函数，指数函数，幂运算，除法，反向传播
    #再加点常数进去看看反向除法他们有没有正常运行
    x1=Value(2.0,label='x1')
    x2=Value(0.0,label='x2')
    w1=-3
    w2=1    
    b=Value(6.8813735870195432,label='b')
    x1w1=x1*w1; x1w1.label='x1*w1'
    x2w2=x2*w2; x2w2.label='x2*w2'
    x1w1x2w2=x1w1+x2w2; x1w1x2w2.label='x1*w1+x2*w2'
    n=x1w1x2w2+b; n.label='n'

    o=n.tanh(); o.label='o'
    o.backward()
    dot=draw_dot(o)
    display(dot)


In [60]:
#接下来做的是神经元的，为了实现神经元部分的，这里要用到pytorch了
import torch
import random
class neuron:
    #神经元类，初始化的时候需要输入神经元的输入个数nin，随机初始化权重和偏置
    def __init__(self, nin):
        self.w = [Value(random.uniform(-1, 1)) for _ in range(nin)]
        self.b = Value(random.uniform(-1, 1))
    #调用的函数call，用来调用的。
    def __call__(self, x):
        act = sum((wi*xi for wi, xi in zip(self.w, x)), self.b)
        out = act.tanh()
        return out
    #记录权重的梯度的，
    def parameters(self):
        return self.w +[self.b]
class layer:
    #层类，初始化的时候需要输入神经元的输入个数nin和输出个数nout，随机初始化权重和偏置，这个层就是很多神经元组成的，一层的东西，
    def __init__(self, nin, nout):
        self.neurons = [neuron(nin) for _ in range(nout)]
    def __call__(self, x):
        out=[neurons(x)for neurons in self.neurons]
        
        return out[0] if len(out)==1 else out
    def parameters(self):
        return [p for n in self.neurons for p in n.parameters()]
class mlp:
    #多层感知机类，初始化的时候需要输入神经元的输入个数nin和输出个数nouts，这个类就是很多层组成的，所以顾名思义就叫多层感知机
    def __init__(self, nin, nouts):
        sz = [nin] + nouts
        self.layers = [layer(sz[i], sz[i+1]) for i in range(len(nouts))]
    def __call__(self, x):
        for layer in self.layers:
            x = layer(x)
        return x
    def parameters(self):
        return [p for layer in self.layers for p in layer.parameters()]



In [73]:
#我先试试Andrej Karpathy给的数据，待会再自己搞个看看,先把数据列出来
xs=[[2.0, 3.0, -1.0], [3.0, -1.0, 0.5], [0.5, 1.0, 1.0], [1.0, 0.4, 0.9]]
ys=[1.0, -1.0, -1.0, 1.0]#期待的输出
n=mlp(3,[4,4,1])

for k in range(30):
    ypred=[n(x)for x in xs]
    loss=sum((yout-ygt)**2 for ygt,yout in zip(ys,ypred))
    #测试一下刚刚写的神经网络有没有正常运行的,就当写一组测试数据一样
    for p in n.parameters():
        p.grad=0.0
    loss.backward()
    for p in n.parameters():
        p.data+=-0.1*p.grad
        
    print(k,loss.data)



0 5.382984039868701
1 3.6883521220353623
2 3.286088895150936
3 2.6331701083077816
4 2.308275127200011
5 2.03862025142841
6 2.16226066488671
7 3.8890681788571455
8 3.8769485735535407
9 3.1944121492199287
10 1.7372859506339107
11 1.3768448580616162
12 2.0359973007125687
13 2.710799703184268
14 3.2860588467982215
15 0.677958651745422
16 0.58897892145776
17 0.671565051395976
18 0.5396251579757735
19 0.35236766378049633
20 0.11089390432825973
21 0.0672150303537088
22 0.057399625504240104
23 0.051304606356088675
24 0.04661895380635643
25 0.042763252141227726
26 0.03949675689812443
27 0.03668222363370342
28 0.03422789450797676
29 0.03206742092524522


In [74]:
ypred

[Value(data=0.9756308980061343),
 Value(data=-0.9008318885431132),
 Value(data=-0.9108333487822243),
 Value(data=0.8830018727759293)]